# Unit 9 — Recursion & Backtracking

How many ways can you choose some of the numbers `4 3 1 2` so they add up to exactly `5`? (There are two: `4+1` and `3+2`.)
Listing them by hand is easy for five numbers, but the number of choices doubles with every extra number.
In this unit, recursion solves one smaller decision at a time, and backtracking tries a choice, explores where it leads, then backs up to try the next one — so we can count or list *every* valid answer.

## Lesson 1 — A Function Can Solve a Smaller Copy

A recursive function is a named function that calls itself.
Every recursive function needs a **base case** that answers the smallest input without another call and a **recursive case** that moves toward that base case.
For factorial, `factorial(0)` is `1`; for a positive `n`, `factorial(n)` is `n * factorial(n - 1)`.

In [ ]:
def solve(data: str) -> str:
    n = int(data.strip())

    def factorial(value):
        if value == 0:
            return 1
        return value * factorial(value - 1)

    return str(factorial(n))

assert solve("5") == "120"
assert solve("0") == "1"

## The Call Stack

When `factorial(3)` calls `factorial(2)`, the first call waits while Python starts the smaller call.
The waiting calls form the **call stack**: `factorial(3)`, then `factorial(2)`, then `factorial(1)`, then the base case `factorial(0)`.
After the base case returns, the calls finish in reverse order.
The input limit must keep this stack shallow enough; recursion is not a safe way to count down from an enormous number.

In [ ]:
def solve(data: str) -> str:
    values = []
    for token in data.split():
        values.append(int(token))

    def recursive_sum(index):
        if index == len(values):
            return 0
        return values[index] + recursive_sum(index + 1)

    return str(recursive_sum(0))

assert solve("4 7 6") == "17"
assert solve("12 30 6") == "48"
assert solve("") == "0"

In [ ]:
def solve(data: str) -> str:
    n = int(data.strip())

    def countdown(value):
        if value == 0:
            return "GO"
        return str(value) + " " + countdown(value - 1)

    return countdown(n)

assert solve("4") == "4 3 2 1 GO"
assert solve("0") == "GO"

## Lesson 2 — Try, Recurse, Undo

Backtracking builds a partial solution one decision at a time.
At each decision, it tries one legal choice, recurses to finish the rest, and then **undoes** that choice before trying another branch.
A safe pop-free pattern is to pass `path + [choice]` into the next call.
That expression makes a new list, so returning automatically leaves the caller's `path` unchanged.

## Include or Skip: Subsets

For each value, a subset search has two choices: include the value or skip it.
The base case arrives after every input position has been decided.
The first recursive call receives a new path containing the value; when it returns, the original path is already restored for the skip branch.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    target = int(tokens[0])
    values = []
    for token in tokens[1:]:
        values.append(int(token))
    n = len(values)

    def search(index, total, path):
        if index == n:
            if total == target:
                return 1
            return 0
        with_value = search(index + 1, total + values[index], path + [values[index]])
        without_value = search(index + 1, total, path)
        return with_value + without_value

    return str(search(0, 0, []))

assert solve("7 2 5 1 6") == "2"
assert solve("5 4 3 1 2") == "2"
assert solve("0") == "1"

## Choose an Unused Value: Permutations

A permutation search tries every value that has not been used yet in the next position.
A boolean list `used` tracks availability: setting `used[choice_index] = True` records that a value is taken, the recursive call receives a new extended path, and setting `used[choice_index] = False` performs the undo.
Without that undo, a value chosen in one branch would incorrectly stay unavailable in later branches.

In [ ]:
def solve(data: str) -> str:
    values = []
    for token in data.split():
        values.append(int(token))
    used = []
    for value in values:
        used.append(False)

    def search(path):
        if len(path) == len(values):
            return 1
        ways = 0
        for choice_index in range(len(values)):
            if used[choice_index]:
                continue
            choice = values[choice_index]
            if len(path) == 0 or abs(path[-1] - choice) >= 2:
                used[choice_index] = True
                ways = ways + search(path + [choice])
                used[choice_index] = False
        return ways

    return str(search([]))

assert solve("1 2 4") == "2"
assert solve("5") == "1"

## Choose in Increasing Index Order: Combinations

For combinations, order does not matter.
Passing `next_index` means a later choice can only come from a later position, so the same group is not counted again in a different order.
The recursion stops when the path has the requested size.

In [ ]:
def solve(data: str) -> str:
    tokens = data.split()
    n = int(tokens[0])
    choose = int(tokens[1])

    def search(next_index, path):
        if len(path) == choose:
            return 1
        ways = 0
        for choice in range(next_index, n):
            ways = ways + search(choice + 1, path + [choice])
        return ways

    return str(search(0, []))

assert solve("5 3") == "10"
assert solve("4 0") == "1"

## Lesson 3 — Parsing a Nested Expression by Recursion

Some inputs are themselves nested, like the scored expression `(2+(3*4))`.
Recursion fits naturally: to evaluate what is inside a pair of parentheses, find the single operator that joins its two sides, then evaluate each side the same way.
The key step is scanning left to right while tracking the **depth** (how many parentheses are still open); the operator that joins the two sides is the one seen at depth zero inside the current pair.
A plain number, with no parentheses, is the base case.

In [ ]:
def solve(data: str) -> str:
    text = data.strip()

    def value(expression):
        if expression[0] != "(":
            return int(expression)
        depth = 0
        split_at = -1
        index = 1
        while index < len(expression) - 1:
            character = expression[index]
            if character == "(":
                depth = depth + 1
            elif character == ")":
                depth = depth - 1
            elif depth == 0 and (character == "+" or character == "*"):
                split_at = index
            index = index + 1
        left = value(expression[1:split_at])
        right = value(expression[split_at + 1:len(expression) - 1])
        if expression[split_at] == "+":
            return left + right
        return left * right

    return str(value(text))

assert solve("(2+(3*4))") == "14"
assert solve("((1+2)*(3+4))") == "21"
assert solve("7") == "7"

## A Backtracking Checklist

First, state exactly what the partial solution means.
Second, write a base case for a complete decision sequence.
Third, try each legal choice and make sure each recursive call moves closer to the base case.
Finally, undo any shared-state change before the next choice; use `path + [choice]` when a fresh path is simplest.
Keep input sizes small because subset and permutation searches can create exponentially many calls.

## Submit the Solver

After `solve(data)` works, a contest submission can use this wrapper.
It is marked `no-exec` because notebook execution has no contest input waiting for it.

In [ ]:
import sys
print(solve(sys.stdin.read()))